In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.sql('select * from ipl.01_raw.ipl_data')

In [0]:
df_final = df.select(
    col("info.dates")[0].alias("match_date").cast("date"),
    col("info.season").alias("season"),
    col("info.event.match_number").alias("match_number").cast("int"),
    col("info.city").alias("city"),
    col("info.venue").alias("venue"),
    col("info.officials.umpires")[0].alias("umpire1"),
    col("info.officials.umpires")[1].alias("umpire2"),
    year(col("info.dates")[0]).alias("match_year").cast("int"),
    month(col("info.dates")[0]).alias("match_month").cast("int"),
    day(col("info.dates")[0]).alias("match_day").cast("int"),
    col("info.outcome.winner").alias("match_winner"),
    col("info.player_of_match")[0].alias("player_of_the_match"),
    col("info.toss.winner").alias("toss_winner"),
    col("info.toss.decision").alias("toss_decision"),
    col("innings")[0].team.alias("team1"),
    when(
        col("info.outcome.result") != "no result",
        col("innings")[1].team
    ).otherwise(
        when(
            col("info.teams")[0] != col("innings")[0].team,
            col("info.teams")[0]
        ).otherwise(col("info.teams")[1])
    ).alias("team2"),
    size(col("innings")).alias("innings_count").cast("int")
).sort("match_date","match_number")
display(df_final)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# Window for ranking within each season ordered by match_date where match_number is null
w = Window.partitionBy("season", col("match_number").isNull()).orderBy("match_date")

df_final = df_final.withColumn(
    "match_number",
    when(
        col("match_number").isNull(),
        (lit(99) * 10 + rank().over(w))
    ).otherwise(col("match_number"))
)

display(df_final)

In [0]:
df_final = df_final.withColumn("match_id", concat(date_format("match_date", "yyyyMMdd"), lpad(col("match_number"), 3, "0")).cast("bigint"))
display(df_final)

In [0]:
df_final.write.mode("overwrite").saveAsTable("ipl.02_enriched.matches")